# Libras Livre — pré-treino contrastivo + fine-tuning em GPU

Executa o código de `computer-vision-model/treino/` no Colab/Kaggle: **ImageNet →
V-LIBRASIL (contrastivo) → MINDS (fine-tuning com avaliação LOSO)**.
A extração MediaPipe continua local, em CPU; não se enviam vídeos para este notebook.

## Antes de começar

1. Ative a GPU no runtime. O notebook aborta sem CUDA para evitar treino pesado em CPU.
2. Prepare **dois pacotes privados**, preservando os nomes e a estrutura:
   - `landmarks-minds.tar.gz` → pasta `landmarks/`, com os 800 `.npy` MINDS (`pessoaM*`).
     Metadados são opcionais no treino normal; não misture V-LIBRASIL nesta pasta.
   - `landmarks-vlibrasil.tar.gz` → pasta `landmarks-pretreino/`, com o corpus V-LIBRASIL
     atual de **4.053 clipes após a exclusão** dos 30 reservados. Inclua, junto de **cada**
     `.npy`, seu sidecar `*.npy.proveniencia.json`. Não empacote somente os `.npy`.
     Sidecars devem vir do pipeline de proveniência; não invente metadados para passar na auditoria.
3. No Colab, envie ambos ao runtime privado; no Kaggle, use entrada e notebook **privados**.
   Alternativamente, aponte os caminhos para armazenamento privado autorizado.
4. O clone precisa conter a implementação de `pretreinar.py --auditar`: audita e sai **sem
   construir modelo ou treinar**. Sem essa flag, a execução para; não remova a auditoria.

**Licença e privacidade:** MINDS declara MIT; V-LIBRASIL é **CC BY-NC-ND**
(não-comercial, sem derivações). A extração de landmarks não elimina essas restrições.
Use apenas em ambiente privado autorizado, respeitando os termos; isto não é autorização
de redistribuição ou uso comercial. **Não publique** vídeos, landmarks, sidecars, pacotes,
checkpoints derivados ou saídas privadas em GitHub, Hugging Face, Kaggle público ou notebook público.
O download ao final é para armazenamento privado do operador, não publicação.

**Limite da avaliação:** os 30 clipes V-LIBRASIL são reservados, sem sobreposição exata
com o corpus. Após o pré-treino, seus articuladores/domínio são conhecidos (inclusive por
validação interna de V03): não medem domínio nem pessoas inéditas. V03 escolhe a época,
não é teste independente. Classes compartilhadas com MINDS são legítimas em transferência
supervisionada; o isolamento exigido é de amostras/origens e das pessoas MINDS de teste.
Generalização no balcão exige coleta própria.

## 1. Ambiente e código

In [ ]:
import os, pathlib, shutil, subprocess, sys

# ORDEM IMPORTA: o editor novo do Kaggle é baseado em Colab, então
# `google.colab` está no sys.modules dele e /content existe. Testar Colab
# primeiro faz o Kaggle se identificar como Colab e cair em files.upload() —
# um seletor de arquivo que ninguém clica numa execução em background.
# /kaggle/working só existe no Kaggle, então ele decide primeiro.
EM_KAGGLE = os.path.exists("/kaggle/working") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
EM_COLAB = not EM_KAGGLE and ("google.colab" in sys.modules or os.path.exists("/content"))
BASE = pathlib.Path("/kaggle/working" if EM_KAGGLE else "/content" if EM_COLAB else ".")
print("ambiente:", "Kaggle" if EM_KAGGLE else "Colab" if EM_COLAB else "local", "| base:", BASE)

URL = "https://github.com/Heitorvazeg/libras-livre-ai-glasses-brasil.git"
BRANCH = "claude/libras-detection-model-53kd30"   # onde vive o código de modelo
REPO = BASE / "libras-livre-ai-glasses-brasil"
TREINO = REPO / "computer-vision-model" / "treino"

def git(*args, repo=None):
    cmd = ["git"] + (["-C", str(repo)] if repo else []) + list(args)
    return subprocess.run(cmd, capture_output=True, text=True)

# Esta célula deixa o código SEMPRE atual, em três situações diferentes:
#   1. não há clone            -> clona a branch certa
#   2. há clone, branch errada -> descarta e reclona (foi o que aconteceu quando
#                                 o clone veio da default e não tinha treino/)
#   3. há clone, branch certa  -> ATUALIZA. Sem isso, reabrir o notebook num
#                                 runtime que ainda vive reaproveita código velho
#                                 e as correções recém-publicadas não chegam.
if REPO.exists() and (git("rev-parse", "--abbrev-ref", "HEAD", repo=REPO).stdout.strip() != BRANCH
                      or not TREINO.is_dir()):
    print("clone existente está na branch errada ou incompleto — refazendo")
    shutil.rmtree(REPO)

if REPO.exists():
    antes = git("rev-parse", "--short", "HEAD", repo=REPO).stdout.strip()
    git("fetch", "--depth", "1", "origin", BRANCH, repo=REPO)
    git("reset", "--hard", f"origin/{BRANCH}", repo=REPO)
    depois = git("rev-parse", "--short", "HEAD", repo=REPO).stdout.strip()
    print(f"repo atualizado: {antes} -> {depois}" if antes != depois else
          f"repo já estava atual ({depois})")
else:
    # --branch no clone: sem isso vem a default (main), que não tem treino/.
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, URL, str(REPO)],
                   check=True)

assert TREINO.is_dir(), f"{TREINO} não existe mesmo após o clone"
print("branch:", git("rev-parse", "--abbrev-ref", "HEAD", repo=REPO).stdout.strip())
print("último commit:", git("log", "-1", "--pretty=%h %s", repo=REPO).stdout.strip())
print("código em:", TREINO)


In [ ]:
import torch
print("torch", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Ative a GPU antes de continuar; não executar treino pesado em CPU.")
print("GPU:", torch.cuda.get_device_name(0))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "scipy"], check=True)

## 2. Dois pacotes privados de landmarks

MINDS vai para `PoC/data/landmarks/` (caminho de `PoC/config.yaml`); V-LIBRASIL,
para `PoC/data/landmarks-pretreino/`, passado explicitamente em `--corpus`.
A extração exige suporte a `tarfile` com `filter="data"`; **não há fallback inseguro**.
Só são aceitos arquivos regulares `.npy` e sidecars na pasta esperada. Pastas de destino
já preenchidas causam erro, para não misturar pacotes com dados antigos.
A checagem local dos sidecars valida presença e JSON não vazio; o esquema, a origem,
os hashes e o isolamento pertencem à auditoria obrigatória da próxima etapa.

In [ ]:
import json
import tarfile
import tempfile

DESTINO = (REPO / "computer-vision-model" / "PoC" / "data").resolve()
DESTINO.mkdir(parents=True, exist_ok=True)
PACOTES = {
    "landmarks-minds.tar.gz": "landmarks",
    "landmarks-vlibrasil.tar.gz": "landmarks-pretreino",
}

if EM_COLAB:
    from google.colab import files
    print("Selecione os DOIS pacotes privados: " + ", ".join(PACOTES))
    enviados = files.upload()
    if set(enviados) != set(PACOTES):
        raise ValueError("Envie exatamente os dois pacotes esperados, com os nomes indicados.")
    origens = {nome: pathlib.Path(nome).resolve() for nome in PACOTES}
    del enviados  # libera as cópias dos pacotes em memória
else:
    # Ajuste para suas entradas PRIVADAS no Kaggle ou caminhos locais autorizados.
    if EM_KAGGLE:
        # PROCURA em vez de exigir um caminho exato. Exigir o slug
        # 'libras-landmarks' foi decisão ruim: o Kaggle deriva o slug do TÍTULO
        # do dataset, então "Libras Landmarks MINDS" vira libras-landmarks-minds
        # e o caminho não bate — falha que só aparece com a GPU já alocada.
        raiz = pathlib.Path("/kaggle/input")
        achados = {n: sorted(raiz.glob(f"*/{n}")) + sorted(raiz.glob(f"*/*/{n}"))
                   for n in PACOTES}
        faltando = [n for n, v in achados.items() if not v]
        if faltando:
            inventario = {d.name: [q.name for q in sorted(d.iterdir())[:8]]
                          for d in sorted(raiz.iterdir())} if raiz.is_dir() else {}
            extraido = any("landmarks" in c for cs in inventario.values() for c in cs)
            raise FileNotFoundError(
                f"Não achei {faltando} em /kaggle/input.\nEncontrado: {inventario}\n"
                + ("O dataset parece já EXTRAÍDO pelo Kaggle. Reenvie o .tar.gz "
                   "dentro de um .zip — o Kaggle descompacta o zip e preserva o "
                   "tar.gz intacto.\n" if extraido else "")
                + "Confira também se o dataset está anexado em Input.")
        origens = {n: v[0] for n, v in achados.items()}
        print("pacotes localizados:", {n: str(v) for n, v in origens.items()})
    else:
        raiz_pacotes = pathlib.Path("~").expanduser()
        origens = {nome: raiz_pacotes / nome for nome in PACOTES}

if not hasattr(tarfile, "data_filter"):
    raise RuntimeError("Atualize o Python: extração exige filter='data', sem fallback.")
for nome, pasta in PACOTES.items():
    if not origens[nome].is_file():
        raise FileNotFoundError(origens[nome])
    alvo = DESTINO / pasta
    if alvo.is_symlink() or (alvo.exists() and (not alvo.is_dir() or any(
        p.name != ".gitkeep" or not p.is_file() or p.is_symlink() for p in alvo.iterdir()
    ))):
        raise RuntimeError(f"Destino já preenchido: {alvo}. Use um runtime/diretório limpo.")

# Valida os dois pacotes numa área temporária antes de instalar qualquer um.
with tempfile.TemporaryDirectory(dir=DESTINO) as temporario:
    staging = pathlib.Path(temporario)
    for nome, pasta in PACOTES.items():
        with tarfile.open(origens[nome], "r:gz") as tar:
            membros = tar.getmembers()
            vistos = set()
            for membro in membros:
                caminho = pathlib.PurePosixPath(membro.name)
                if caminho.is_absolute() or ".." in caminho.parts or not caminho.parts or caminho.parts[0] != pasta:
                    raise ValueError(f"Caminho inesperado no pacote {nome}: {membro.name}")
                if caminho in vistos:
                    raise ValueError(f"Membro duplicado: {membro.name}")
                vistos.add(caminho)
                if membro.isdir() and len(caminho.parts) == 1:
                    continue
                if not (membro.isfile() and len(caminho.parts) == 2
                        and caminho.name.endswith((".npy", ".npy.proveniencia.json"))):
                    raise ValueError(f"Só landmarks/sidecars regulares são permitidos: {membro.name}")
            tar.extractall(staging, members=membros, filter="data")

        npys = sorted((staging / pasta).glob("*.npy"))
        prefixo = "pessoaM" if pasta == "landmarks" else "pessoaV"
        if not npys or any(not p.name.startswith(prefixo) for p in npys):
            raise ValueError(f"Pacote vazio ou fonte incorreta: {nome}")
        for npy in npys:
            sidecar = npy.with_name(npy.name + ".proveniencia.json")
            if pasta == "landmarks-pretreino" or sidecar.exists():
                if not sidecar.is_file():
                    raise FileNotFoundError(f"Proveniência obrigatória ausente: {sidecar.name}")
                meta = json.loads(sidecar.read_text(encoding="utf-8"))
                if not isinstance(meta, dict) or not meta:
                    raise ValueError(f"Sidecar deve conter um objeto JSON não vazio: {sidecar.name}")
        for sidecar in (staging / pasta).glob("*.npy.proveniencia.json"):
            if not sidecar.with_name(sidecar.name.removesuffix(".proveniencia.json")).is_file():
                raise ValueError(f"Sidecar sem landmark correspondente: {sidecar.name}")
        pessoas = sorted({p.name.split("_")[0] for p in npys})
        if pasta == "landmarks-pretreino" and "pessoaV03" not in pessoas:
            raise ValueError("V03 não está no corpus; não é possível reservar a validação explícita.")
        print(f"{pasta}: {len(npys)} clipes | pessoas: {', '.join(pessoas)}")

    for pasta in PACOTES.values():
        alvo = DESTINO / pasta
        alvo.mkdir(exist_ok=True)
        # Preserva o .gitkeep do clone limpo; nunca remove dados de uma execução antiga.
        for item in (staging / pasta).iterdir():
            shutil.move(str(item), str(alvo / item.name))

CORPUS = DESTINO / "landmarks-pretreino"

## 3. Auditoria → pré-treino contrastivo → fine-tuning MINDS

O experimento principal usa **ResNet-18**, com `--objetivo contrastivo` e
`--pessoa-val V03` explícitos. Primeiro executa `--auditar` em CPU: deve encerrar sem
modelo. Qualquer falha interrompe a execução (`check=True`); não contorne a auditoria.
Só depois rodam os testes sintéticos e o pré-treino na GPU.

O fine-tuning usa apenas `--fontes minds` e inicializa **cada rodada LOSO** com o
checkpoint V-LIBRASIL. O teste MINDS permanece fora do pré-treino e do ajuste daquela
rodada. Ao fim, uma execução `--final` separada gera o checkpoint com todos os MINDS:
esse modelo final **não tem métrica independente**, a estimativa vem da LOSO anterior.

Cada execução cria uma pasta privada única, com saídas distintas para pré-treino,
fine-tuning LOSO, modelo final e baselines opcionais (sem pré-treino V-LIBRASIL).
As diferenças devem ser comparadas sob a mesma configuração/partições; ~1,7 pp de
variação já foi observado. O DTW da PoC (10 classes) não é comparação direta com 20 classes.

In [ ]:
from datetime import datetime
from uuid import uuid4

TREINO = TREINO.resolve()
EXP = BASE.resolve() / "experimentos-privados" / (
    datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:8]
 )
EXP.mkdir(parents=True, exist_ok=False)
SAIDA_PRE = EXP / "pretreino-vlibrasil-contrastivo"
SAIDA_FT = EXP / "finetuning-minds-loso"
SAIDA_FINAL = EXP / "finetuning-minds-final"
CHECKPOINT = SAIDA_PRE / "backbone_resnet.pt"

PRE_ARGS = [
    "--corpus", str(CORPUS), "--arquitetura", "resnet",
    "--objetivo", "contrastivo", "--pessoa-val", "V03",
    "--epocas", "15", "--lr", "1e-4", "--batch", "64",
    "--p-classes", "32", "--k-exemplos", "2", "--semente", "0",
    "--saida", str(SAIDA_PRE),
 ]

# Obrigatório: a flag deve auditar e sair antes de construir qualquer modelo.
# Se o clone ainda não tiver --auditar, check=True aborta; não há alternativa permissiva.
subprocess.run(
    [sys.executable, "pretreinar.py", *PRE_ARGS, "--dispositivo", "cpu", "--auditar"],
    cwd=TREINO, check=True,
 )

# Validação sintética do encanamento, só depois da auditoria aprovada.
subprocess.run([sys.executable, "selftest.py"], cwd=TREINO, check=True)
print("Saídas privadas desta execução:", EXP)

In [ ]:
# Estágio 1: pré-treino contrastivo V-LIBRASIL, nunca MINDS.
subprocess.run(
    [sys.executable, "pretreinar.py", *PRE_ARGS, "--dispositivo", "cuda"],
    cwd=TREINO, check=True,
 )
if not CHECKPOINT.is_file() or not CHECKPOINT.with_suffix(".json").is_file():
    raise FileNotFoundError("Pré-treino não produziu backbone_resnet.pt e seu JSON.")
print("Backbone que será transferido:", CHECKPOINT)

In [ ]:
# Estágio 2: fine-tuning supervisionado MINDS, todas as rodadas LOSO.
FT_ARGS = [
    "--arquitetura", "resnet", "--fontes", "minds", "--dispositivo", "cuda",
    "--epocas", "30", "--lr", "1e-4", "--batch", "64", "--agendador", "nenhum",
 ]
if not CHECKPOINT.is_file():
    raise FileNotFoundError(CHECKPOINT)
subprocess.run(
    [sys.executable, "treinar.py", *FT_ARGS, "--folds", "0",
     "--inicializar", str(CHECKPOINT), "--saida", str(SAIDA_FT)],
    cwd=TREINO, check=True,
 )

# Checkpoint final separado: todos os MINDS, sem alegar nova métrica de teste.
subprocess.run(
    [sys.executable, "treinar.py", *FT_ARGS, "--final",
     "--inicializar", str(CHECKPOINT), "--saida", str(SAIDA_FINAL)],
    cwd=TREINO, check=True,
 )
if not (SAIDA_FINAL / "modelo_final.pt").is_file():
    raise FileNotFoundError("Fine-tuning final não produziu modelo_final.pt.")

In [ ]:
# Baselines OPCIONAIS e separados — não fazem parte do caminho principal.
RODAR_BASELINE_RESNET = False  # ImageNet → MINDS, sem V-LIBRASIL
RODAR_BASELINE_GCN = False     # GCN do zero; orçamento próprio, não ablação da ResNet

if RODAR_BASELINE_RESNET:
    subprocess.run(
        [sys.executable, "treinar.py", *FT_ARGS, "--folds", "0",
         "--saida", str(EXP / "baseline-resnet-imagenet-minds")],
        cwd=TREINO, check=True,
    )

if RODAR_BASELINE_GCN:
    subprocess.run(
        [sys.executable, "treinar.py", "--arquitetura", "gcn",
         "--fontes", "minds", "--dispositivo", "cuda", "--folds", "0",
         "--epocas", "120", "--lr", "1e-3", "--batch", "32",
         "--agendador", "cosseno", "--saida", str(EXP / "baseline-gcn-minds")],
        cwd=TREINO, check=True,
    )

## 4. Baixar todos os artefatos do experimento — uso privado

Antes de encerrar o runtime, baixe o arquivo completo: inclui **backbone `.pt`, JSON
de metadados, checkpoint final, relatórios, matrizes e qualquer outro artefato gerado**
nos diretórios desta execução, inclusive baselines se ativados. Não é só o relatório.
Landmarks e pacotes de entrada ficam fora desse arquivo.

Colab: download direto para a máquina do operador. Kaggle/local: link para download
privado; não publique o notebook nem transforme a saída em dataset público.
Os checkpoints derivados de V-LIBRASIL também devem permanecer privados, sujeitos
à análise de licença antes de qualquer distribuição ou uso comercial.

In [ ]:
for rel in sorted(EXP.rglob("relatorio.md")):
    print("=" * 70, "\n", rel.relative_to(EXP))
    print(rel.read_text(encoding="utf-8")[:1500])

artefatos = sorted(p for p in EXP.rglob("*") if p.is_file())
if not artefatos:
    raise RuntimeError("Nenhum artefato para baixar.")
for artefato in artefatos:
    print(artefato.relative_to(EXP), "|", artefato.stat().st_size, "bytes")

# Arquivo fora de EXP para não incluir a si próprio; preserva todos os artefatos.
arquivo = EXP.parent / f"{EXP.name}.tar.gz"
with tarfile.open(arquivo, "w:gz") as tar:
    tar.add(EXP, arcname=EXP.name)
print("Arquivo completo PRIVADO:", arquivo)

if EM_COLAB:
    from google.colab import files
    files.download(str(arquivo))
else:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(arquivo, pathlib.Path.cwd())))